In [1]:
from __future__ import annotations

import csv
import html
import json
import re
import subprocess
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin

BASE_URL = "https://fraser.stlouisfed.org"
TITLE_SLUG = "g6-debits-deposit-turnover-commercial-banks-3954"
DECADE_URLS = {
    decade: f"{BASE_URL}/title/{TITLE_SLUG}"
    for decade in ("1970s", "1980s", "1990s")
}
START_DATES = {"1970s": datetime.strptime("October 13, 1977", "%B %d, %Y").date()}
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "deposit_turnover_SQL_database.ipynb").exists():
    PROJECT_ROOT /= "SQL database: Deposit Turnover by Type, 1977–1996"
OUTPUT_DIR = PROJECT_ROOT / "fraser_g6_issues"
CATALOG_PATH = PROJECT_ROOT / "fraser_g6_issue_catalog.json"
REQUEST_DELAY_SECONDS = 0.35
TIMEOUT_SECONDS = 60
MAX_RETRIES = 4
USER_AGENT = "Personal-Macro-Research/1.0 (FRASER historical-document downloader)"


def fetch_bytes(url: str) -> tuple[bytes, str]:
    command = [
        "curl", "--http1.1", "--fail", "--location", "--silent", "--show-error",
        "--retry", str(MAX_RETRIES), "--retry-all-errors", "--retry-delay", "2",
        "--connect-timeout", "30", "--max-time", str(TIMEOUT_SECONDS),
        "--user-agent", USER_AGENT, url,
    ]
    try:
        completed = subprocess.run(command, check=True, capture_output=True)
    except FileNotFoundError as error:
        raise RuntimeError("curl is required but was not found on this computer") from error
    except subprocess.CalledProcessError as error:
        detail = error.stderr.decode("utf-8", errors="replace").strip()
        raise RuntimeError(f"FRASER request failed: {url} ({detail})") from error
    content_type = "application/pdf" if completed.stdout.startswith(b"%PDF-") else "text/html"
    return completed.stdout, content_type


def discover_issues(decade_url: str, decade: str) -> list[dict]:
    if not CATALOG_PATH.exists():
        raise FileNotFoundError(f"Missing local issue catalog: {CATALOG_PATH.resolve()}")
    browse_data = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
    if decade not in browse_data:
        raise RuntimeError(f"Local catalog did not contain the {decade} issue group")

    issues = []
    for item in browse_data[decade]:
        issue_date = datetime.strptime(item["name"].split(" :", 1)[0], "%B %d, %Y").date()
        if issue_date >= START_DATES.get(decade, datetime.min.date()):
            issues.append({
                "decade": decade,
                "item_id": item["id"],
                "issue_name": item["name"],
                "issue_date": issue_date.isoformat(),
                "issue_url": urljoin(BASE_URL, item["url"]),
            })
    return issues


def find_pdf_url(issue: dict) -> str:
    compact_date = issue["issue_date"].replace("-", "")
    return f"{BASE_URL}/files/docs/releases/g6comm/g6_{compact_date}.pdf"


def valid_pdf(path: Path) -> bool:
    return path.is_file() and path.stat().st_size > 4 and path.read_bytes()[:5] == b"%PDF-"


def download_issue(issue: dict) -> dict:
    decade_dir = OUTPUT_DIR / issue["decade"]
    decade_dir.mkdir(parents=True, exist_ok=True)
    safe_name = re.sub(r"[^A-Za-z0-9._-]+", "_", issue["issue_name"]).strip("_")
    destination = decade_dir / f'{issue["issue_date"]}_{issue["item_id"]}_{safe_name}.pdf'

    if valid_pdf(destination):
        return {**issue, "pdf_url": "", "file": str(destination), "status": "skipped_existing"}

    pdf_url = find_pdf_url(issue)
    time.sleep(REQUEST_DELAY_SECONDS)
    payload, content_type = fetch_bytes(pdf_url)
    if not payload.startswith(b"%PDF-"):
        raise RuntimeError(f"Expected a PDF from {pdf_url}, received {content_type}")

    temporary = destination.with_suffix(".pdf.part")
    temporary.write_bytes(payload)
    temporary.replace(destination)
    return {**issue, "pdf_url": pdf_url, "file": str(destination), "status": "downloaded"}


def write_manifest(rows: list[dict]) -> Path:
    manifest_path = OUTPUT_DIR / "manifest.csv"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    fields = ["decade", "item_id", "issue_name", "issue_date", "issue_url", "pdf_url", "file", "status"]
    with manifest_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return manifest_path


all_issues = []
for decade, decade_url in DECADE_URLS.items():
    discovered = discover_issues(decade_url, decade)
    all_issues.extend(discovered)
    print(f"{decade}: discovered {len(discovered)} eligible issues")
    time.sleep(REQUEST_DELAY_SECONDS)

results = []
for number, issue in enumerate(all_issues, start=1):
    try:
        result = download_issue(issue)
    except Exception as error:
        result = {**issue, "pdf_url": "", "file": "", "status": f"error: {error}"}
    results.append(result)
    print(f"[{number:>3}/{len(all_issues)}] {issue['issue_name']}: {result['status']}")
    write_manifest(results)
    time.sleep(REQUEST_DELAY_SECONDS)

manifest = write_manifest(results)
downloaded = sum(row["status"] == "downloaded" for row in results)
skipped = sum(row["status"] == "skipped_existing" for row in results)
errors = sum(row["status"].startswith("error:") for row in results)
print(f"Done: {downloaded} downloaded, {skipped} already present, {errors} errors.")
print(f"Manifest: {manifest.resolve()}")


1970s: discovered 26 eligible issues
1980s: discovered 117 eligible issues
1990s: discovered 82 eligible issues
[  1/225] October 13, 1977: downloaded
[  2/225] November 10, 1977: downloaded
[  3/225] December 8, 1977: downloaded
[  4/225] January 11, 1978: downloaded
[  5/225] February 9, 1978: downloaded
[  6/225] March 22, 1978: downloaded
[  7/225] April 13, 1978: downloaded
[  8/225] May 16, 1978: downloaded
[  9/225] June 5, 1978: downloaded
[ 10/225] July 20, 1978: downloaded
[ 11/225] August 2, 1978: downloaded
[ 12/225] September 11, 1978: downloaded
[ 13/225] October 13, 1978: downloaded
[ 14/225] November 14, 1978: downloaded
[ 15/225] December 12, 1978: downloaded
[ 16/225] January 9, 1979: downloaded
[ 17/225] February 15, 1979: downloaded
[ 18/225] April 24, 1979: downloaded
[ 19/225] May 11, 1979: downloaded
[ 20/225] June 13, 1979: downloaded
[ 21/225] July 12, 1979: downloaded
[ 22/225] August 10, 1979: downloaded
[ 23/225] September 13, 1979: downloaded
[ 24/225] Octo

In [1]:
# Detect persistent formatting and operational-definition eras across the downloaded releases.
# The first run installs pypdf if needed and caches extracted PDF features.
import hashlib
import csv
import json
import math
import re
import statistics
import subprocess
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path

try:
    from pypdf import PdfReader
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "pypdf"], check=True)
    from pypdf import PdfReader

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "deposit_turnover_SQL_database.ipynb").exists():
    PROJECT_ROOT /= "SQL database: Deposit Turnover by Type, 1977–1996"
PDF_ROOT = PROJECT_ROOT / "fraser_g6_issues"
FEATURE_CACHE = PDF_ROOT / "era_detection_features.json"
ERA_OUTPUT = PDF_ROOT / "detected_eras.csv"
WINDOW = 3                 # releases compared on each side of a candidate boundary
BOUNDARY_THRESHOLD = 0.30  # lower this to find more candidate eras; raise it for fewer
MIN_BOUNDARY_GAP = 2       # suppress competing boundaries within this many releases

DEFINITION_CONCEPTS = {
    "demand deposits": ("demand deposit",),
    "ATS accounts": ("automatic transfer service", "ats account"),
    "NOW accounts": ("negotiable order of withdrawal", "now account"),
    "savings deposits": ("savings deposit",),
    "MMDA accounts": ("money market deposit account", "mmda"),
    "Super NOW accounts": ("super now",),
    "other checkable deposits": ("other checkable deposit",),
    "IPC deposits": ("individuals partnerships and corporations", "ipc deposit"),
    "annual-rate turnover": ("annual rate of turnover", "annual turnover rate"),
    "seasonal adjustment": ("seasonally adjusted", "seasonal adjustment"),
    "reporting panel": ("reporting bank", "sample bank", "reporting institution"),
    "FR 2573 reporting": ("fr 2573",),
}


def normalized_words(text):
    text = text.lower().replace("&", " and ")
    text = re.sub(r"[^a-z]+", " ", text)
    return [word for word in text.split() if len(word) > 2]


def shingles(words, size=3):
    return {" ".join(words[index:index + size]) for index in range(max(0, len(words) - size + 1))}


def jaccard(left, right):
    union = left | right
    return len(left & right) / len(union) if union else 1.0


def release_date(path):
    return datetime.strptime(path.name[:10], "%Y-%m-%d").date()


def extract_features(path):
    reader = PdfReader(path)
    page_text = [(page.extract_text() or "") for page in reader.pages]
    full_text = "\n".join(page_text)
    words = normalized_words(full_text)
    first_page_words = normalized_words(page_text[0] if page_text else "")
    page = reader.pages[0] if reader.pages else None
    width = round(float(page.mediabox.width), 1) if page else 0.0
    height = round(float(page.mediabox.height), 1) if page else 0.0
    normalized_text = " ".join(words)
    definitions = {
        concept: any(phrase in normalized_text for phrase in phrases)
        for concept, phrases in DEFINITION_CONCEPTS.items()
    }
    return {
        "path": str(path),
        "date": release_date(path).isoformat(),
        "pages": len(reader.pages),
        "width": width,
        "height": height,
        "characters": len(full_text),
        "document_shingles": sorted(shingles(words)),
        "first_page_shingles": sorted(shingles(first_page_words)),
        "definitions": definitions,
    }


pdf_paths = sorted(PDF_ROOT.glob("*/*.pdf"), key=release_date)
if not pdf_paths:
    raise FileNotFoundError(f"No PDFs found under {PDF_ROOT.resolve()}")

cache = {}
if FEATURE_CACHE.exists():
    cache = json.loads(FEATURE_CACHE.read_text(encoding="utf-8"))

features = []
for number, path in enumerate(pdf_paths, 1):
    stat = path.stat()
    cache_key = f"{path}:{stat.st_size}:{stat.st_mtime_ns}"
    if cache_key not in cache:
        cache[cache_key] = extract_features(path)
    features.append(cache[cache_key])
    if number % 25 == 0 or number == len(pdf_paths):
        print(f"Extracted/cached {number}/{len(pdf_paths)} releases")
FEATURE_CACHE.write_text(json.dumps(cache), encoding="utf-8")


def window_signature(rows):
    document_counts = Counter()
    first_page_counts = Counter()
    for row in rows:
        document_counts.update(row["document_shingles"])
        first_page_counts.update(row["first_page_shingles"])
    persistence = max(1, math.ceil(len(rows) / 2))
    return (
        {item for item, count in document_counts.items() if count >= persistence},
        {item for item, count in first_page_counts.items() if count >= persistence},
    )


def prevalence(rows, concept):
    return sum(row["definitions"][concept] for row in rows) / len(rows)


boundary_scores = []
for index in range(WINDOW, len(features) - WINDOW + 1):
    left = features[index - WINDOW:index]
    right = features[index:index + WINDOW]
    left_doc, left_first = window_signature(left)
    right_doc, right_first = window_signature(right)
    document_change = 1.0 - jaccard(left_doc, right_doc)
    first_page_change = 1.0 - jaccard(left_first, right_first)
    page_change = min(1.0, abs(statistics.median(row["pages"] for row in left) - statistics.median(row["pages"] for row in right)) / 4.0)
    definition_change = statistics.mean(
        abs(prevalence(left, concept) - prevalence(right, concept))
        for concept in DEFINITION_CONCEPTS
    )
    score = 0.30 * document_change + 0.30 * first_page_change + 0.15 * page_change + 0.25 * definition_change
    boundary_scores.append((index, score))

candidates = [(index, score) for index, score in boundary_scores if score >= BOUNDARY_THRESHOLD]
selected = []
for index, score in sorted(candidates, key=lambda pair: pair[1], reverse=True):
    if all(abs(index - chosen_index) > MIN_BOUNDARY_GAP for chosen_index, _ in selected):
        selected.append((index, score))
selected.sort()


def boundary_reason(index):
    left = features[max(0, index - WINDOW):index]
    right = features[index:min(len(features), index + WINDOW)]
    reasons = []
    left_pages = statistics.median(row["pages"] for row in left)
    right_pages = statistics.median(row["pages"] for row in right)
    if left_pages != right_pages:
        reasons.append(f"median pages {left_pages:g} -> {right_pages:g}")
    changed = [
        concept for concept in DEFINITION_CONCEPTS
        if abs(prevalence(left, concept) - prevalence(right, concept)) >= 0.67
    ]
    if changed:
        reasons.append("definition terms: " + ", ".join(changed))
    if not reasons:
        reasons.append("persistent text/table-layout change")
    return "; ".join(reasons)


cuts = [0] + [index for index, _ in selected] + [len(features)]
score_by_index = dict(selected)
eras = []
for era_number, (start, end) in enumerate(zip(cuts, cuts[1:]), 1):
    rows = features[start:end]
    median_pages = statistics.median(row["pages"] for row in rows)
    midpoint = (len(rows) - 1) / 2
    representative = min(
        enumerate(rows),
        key=lambda pair: (abs(pair[1]["pages"] - median_pages), abs(pair[0] - midpoint)),
    )[1]
    boundary_score = score_by_index.get(start)
    eras.append({
        "era": era_number,
        "first_release": rows[0]["date"],
        "last_release": rows[-1]["date"],
        "release_count": len(rows),
        "representative_release": representative["date"],
        "representative_file": representative["path"],
        "boundary_score": "" if boundary_score is None else round(boundary_score, 3),
        "boundary_reason": "start of collection" if start == 0 else boundary_reason(start),
    })

with ERA_OUTPUT.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=eras[0].keys())
    writer.writeheader()
    writer.writerows(eras)

low_text = sum(row["characters"] < 500 for row in features)
print(f"\nDetected {len(eras)} candidate eras from {len(features)} releases.")
print(f"Low-text/OCR-poor releases: {low_text}. Results saved to {ERA_OUTPUT.resolve()}\n")
print("| Era | First | Last | N | Representative | Boundary evidence |")
print("|---:|:---|:---|---:|:---|:---|")
for era in eras:
    print(f"| {era['era']} | {era['first_release']} | {era['last_release']} | {era['release_count']} | {era['representative_release']} | {era['boundary_reason']} |")

print("\nTreat these as candidate boundaries: manually inspect each representative and the releases immediately before/after every boundary.")


Extracted/cached 25/225 releases
Extracted/cached 50/225 releases
Extracted/cached 75/225 releases
Extracted/cached 100/225 releases
Extracted/cached 125/225 releases
Extracted/cached 150/225 releases
Extracted/cached 175/225 releases
Extracted/cached 200/225 releases
Extracted/cached 225/225 releases

Detected 19 candidate eras from 225 releases.
Low-text/OCR-poor releases: 0. Results saved to /Users/danie/Personal-Macro-Research/fraser_g6_issues/detected_eras.csv

| Era | First | Last | N | Representative | Boundary evidence |
|---:|:---|:---|---:|:---|:---|
| 1 | 1977-10-13 | 1980-07-09 | 33 | 1979-02-15 | start of collection |
| 2 | 1980-08-14 | 1981-03-26 | 9 | 1980-11-10 | median pages 1 -> 2; definition terms: NOW accounts |
| 3 | 1981-04-07 | 1981-08-13 | 5 | 1981-06-08 | persistent text/table-layout change |
| 4 | 1981-09-15 | 1982-01-19 | 5 | 1981-11-12 | persistent text/table-layout change |
| 5 | 1982-02-16 | 1982-03-17 | 3 | 1982-02-23 | persistent text/table-layout change

In [ ]:
# Deterministic G.6 spatial-OCR extraction pipeline. Safe to rerun.\n
from pathlib import Path
import runpy
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "deposit_turnover_SQL_database.ipynb").exists():
    PROJECT_ROOT /= "SQL database: Deposit Turnover by Type, 1977–1996"
pipeline_path = PROJECT_ROOT / "g6_spatial_extraction_pipeline.py"
if not pipeline_path.exists():
    raise FileNotFoundError(f"Pipeline module not found: {pipeline_path.resolve()}")
runpy.run_path(str(pipeline_path), run_name="__main__")